### Résumé statistique
volume total, ventes habituelles par jour (moyenne, médiane), dispersion (écart-type), part de jours sans vente. Le coefficient de variation (écart-type ÷ moyenne) mesure la régularité : plus il est haut, plus les ventes sont irrégulières, donc difficiles à prévoir.

In [1]:
import pandas as pd
 
d = pd.read_csv("data/clean/ventes_journalier.csv", encoding="utf-8-sig", parse_dates=["date"])

d = d[d["journee_complete"]]                       # on ecarte les 2 journees tronquees

groupes = [c for c in d.columns if c.endswith(")")]  # les 8 colonnes "Nom (code)"
 
resume = pd.DataFrame({

    "total": d[groupes].sum(),

    "moyenne_jour": d[groupes].mean(),

    "mediane_jour": d[groupes].median(),

    "ecart_type": d[groupes].std(),

    "jours_a_zero_%": 100 * (d[groupes] == 0).mean(),

})

resume["coef_variation"] = resume["ecart_type"] / resume["moyenne_jour"]

print(resume.round(2))

                                total  moyenne_jour  mediane_jour  ecart_type  \
AINS acétiques (M01AB)       10723.66          5.10          5.00        2.71   
AINS propioniques (M01AE)     8325.45          3.96          3.68        2.12   
Aspirine et dérivés (N02BA)   8185.76          3.89          3.50        2.34   
Paracétamol (N02BE)          64074.88         30.45         27.00       15.83   
Anxiolytiques (N05B)         18586.01          8.83          8.00        5.53   
Somnifères (N05C)             1247.00          0.59          0.00        1.10   
Antiasthmatiques (R03)       11743.08          5.58          4.00        6.49   
Antihistaminiques (R06)       6162.48          2.93          2.00        2.42   

                             jours_a_zero_%  coef_variation  
AINS acétiques (M01AB)                 0.62            0.53  
AINS propioniques (M01AE)              0.48            0.54  
Aspirine et dérivés (N02BA)            2.57            0.60  
Paracétamol (N02BE)   

### Part des ventes par heure
Répartition des ventes sur la journée, en %, sur les heures d'ouverture et les jours complets.

In [2]:
h = pd.read_csv("data/clean/ventes_horaire.csv", encoding="utf-8-sig", parse_dates=["date"])
h = h[h["heure_ouverture"] & h["date"].dt.normalize().isin(d["date"])]
par_heure = h.groupby("heure")[groupes].mean()
part_heure = 100 * par_heure / par_heure.sum()
display(part_heure.round(1).T)

heure,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
AINS acétiques (M01AB),0.3,5.3,7.1,8.2,9.2,8.8,7.7,8.1,6.1,5.6,5.8,7.0,8.3,7.6,5.0,0.0
AINS propioniques (M01AE),0.5,4.0,5.6,6.7,8.1,7.8,6.7,7.1,6.1,6.4,7.3,8.1,9.8,10.1,5.5,0.0
Aspirine et dérivés (N02BA),0.3,5.4,8.7,10.4,10.4,8.9,8.2,6.5,5.7,5.0,5.6,7.6,7.1,6.7,3.4,0.0
Paracétamol (N02BE),0.5,4.4,5.8,6.6,7.8,8.1,6.9,6.7,6.7,6.7,6.9,8.2,9.5,9.5,5.7,0.1
Anxiolytiques (N05B),0.3,5.7,7.8,9.9,10.8,10.2,9.3,7.2,6.3,5.8,5.5,6.2,6.3,5.5,3.1,0.0
Somnifères (N05C),0.3,4.0,5.2,7.5,7.8,9.5,11.1,7.6,8.8,7.1,6.3,8.4,6.9,7.1,2.4,0.0
Antiasthmatiques (R03),0.5,5.7,6.3,7.4,7.0,5.8,6.6,6.4,5.8,5.1,6.9,9.6,10.6,10.6,5.6,0.1
Antihistaminiques (R06),0.3,4.6,6.1,7.7,7.9,8.2,7.7,6.6,6.1,5.3,5.6,8.4,9.5,9.8,6.2,0.0


### Indice par jour de la semaine
100 = un jour moyen ; au-dessus, c'est un jour fort pour ce groupe.

In [3]:
ordre = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
par_jour = d.groupby("jour_semaine")[groupes].mean().loc[ordre]
indice_jour = 100 * par_jour / d[groupes].mean()
display(indice_jour.round(0).T)

jour_semaine,lundi,mardi,mercredi,jeudi,vendredi,samedi,dimanche
AINS acétiques (M01AB),101.0,100.0,93.0,90.0,95.0,114.0,106.0
AINS propioniques (M01AE),96.0,91.0,94.0,93.0,95.0,111.0,119.0
Aspirine et dérivés (N02BA),101.0,109.0,95.0,96.0,100.0,109.0,90.0
Paracétamol (N02BE),98.0,94.0,93.0,94.0,96.0,112.0,114.0
Anxiolytiques (N05B),105.0,108.0,114.0,99.0,112.0,97.0,65.0
Somnifères (N05C),103.0,121.0,112.0,90.0,129.0,92.0,53.0
Antiasthmatiques (R03),108.0,98.0,98.0,97.0,104.0,100.0,94.0
Antihistaminiques (R06),99.0,105.0,92.0,91.0,95.0,112.0,107.0


### Indice par mois
La saisonnalité : quels mois sont forts ou faibles pour chaque groupe.

In [4]:
m = pd.read_csv("data/clean/ventes_mensuel.csv", encoding="utf-8-sig", parse_dates=["date_fin_mois"])
m = m[m["mois_complet"]]
par_mois = m.groupby(m["date_fin_mois"].dt.month)[groupes].mean()
indice_mois = 100 * par_mois / par_mois.mean()
indice_mois.index.name = "mois"
display(indice_mois.round(0).T)

mois,1,2,3,4,5,6,7,8,9,10,11,12
AINS acétiques (M01AB),110.0,90.0,98.0,101.0,97.0,91.0,104.0,108.0,97.0,99.0,101.0,105.0
AINS propioniques (M01AE),126.0,108.0,100.0,92.0,95.0,88.0,97.0,93.0,98.0,99.0,94.0,108.0
Aspirine et dérivés (N02BA),114.0,112.0,105.0,96.0,98.0,95.0,91.0,91.0,86.0,106.0,106.0,100.0
Paracétamol (N02BE),154.0,109.0,98.0,85.0,76.0,66.0,64.0,68.0,102.0,139.0,103.0,136.0
Anxiolytiques (N05B),115.0,93.0,95.0,90.0,87.0,88.0,96.0,105.0,103.0,109.0,104.0,115.0
Somnifères (N05C),121.0,85.0,108.0,81.0,106.0,98.0,83.0,106.0,87.0,87.0,113.0,126.0
Antiasthmatiques (R03),139.0,110.0,112.0,101.0,90.0,75.0,53.0,55.0,76.0,131.0,108.0,149.0
Antihistaminiques (R06),56.0,54.0,113.0,168.0,179.0,152.0,119.0,95.0,88.0,70.0,54.0,52.0


### Carte heure × jour
Pourquoi : une seule grille montre à quels créneaux de la semaine la pharmacie est la plus sollicitée, utile pour décider des effectifs.

In [5]:
h["jour"] = h["date"].dt.dayofweek.map(dict(enumerate(ordre)))
hm = h.groupby(["jour", "heure"])[groupes].mean().sum(axis=1).unstack().loc[ordre]
display(hm.round(1).style.background_gradient(axis=None))

heure,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
jour,,,,,,,,,,,,,,,,
lundi,0.300000,2.900000,3.600000,4.200000,5.000000,4.200000,4.400000,3.900000,4.000000,3.700000,4.300000,4.900000,5.900000,5.700000,4.800000,0.000000
mardi,0.400000,3.000000,3.800000,4.200000,4.700000,4.500000,4.000000,4.100000,3.500000,3.900000,3.600000,4.900000,5.800000,5.700000,4.300000,0.000000
mercredi,0.400000,3.100000,3.600000,4.000000,4.800000,4.200000,4.300000,4.000000,3.600000,3.300000,3.600000,4.700000,5.600000,5.600000,4.300000,0.000000
jeudi,0.200000,3.200000,3.200000,4.300000,4.400000,4.500000,3.800000,3.900000,3.400000,3.600000,3.900000,4.800000,5.100000,5.500000,4.100000,0.100000
vendredi,0.400000,3.000000,3.900000,4.400000,4.600000,4.800000,4.400000,4.100000,4.200000,3.700000,3.900000,4.500000,5.100000,5.600000,4.300000,0.000000
samedi,0.300000,3.300000,4.500000,5.300000,6.300000,6.200000,5.600000,4.900000,4.400000,4.100000,4.500000,5.300000,5.800000,6.000000,0.100000,0.000000
dimanche,0.000000,2.300000,4.900000,6.200000,6.400000,7.200000,5.400000,4.500000,4.500000,4.000000,4.100000,4.900000,4.900000,3.300000,0.000000,0.000000


### Évolution 2015 à 2018
Tendance de fond par groupe, sur des années complètes.

In [6]:
annuel = d[d["annee"].between(2015, 2018)].groupby("annee")[groupes].sum()
evolution = 100 * (annuel.loc[2018] / annuel.loc[2015] - 1)
display(annuel.round(0).T.assign(evolution_2015_2018_pct=evolution.round(1)))

annee,2015,2016,2017,2018,evolution_2015_2018_pct
AINS acétiques (M01AB),1914.0,2131.0,1852.0,1827.0,-4.5
AINS propioniques (M01AE),1518.0,1611.0,1404.0,1408.0,-7.2
Aspirine et dérivés (N02BA),1622.0,1673.0,1229.0,1140.0,-29.7
Paracétamol (N02BE),12084.0,13544.0,9484.0,11517.0,-4.7
Anxiolytiques (N05B),2988.0,3158.0,2356.0,3302.0,10.5
Somnifères (N05C),159.0,213.0,166.0,241.0,51.6
Antiasthmatiques (R03),1772.0,2153.0,1898.0,2706.0,52.7
Antihistaminiques (R06),1001.0,1076.0,974.0,1212.0,21.1


Marge de sécurité du stock

In [7]:
stock = pd.DataFrame({"moyenne_jour": d[groupes].mean(),
                      "p95_jour": d[groupes].quantile(.95),
                      "max_jour": d[groupes].max()})
stock["p95_sur_moyenne"] = (stock["p95_jour"] / stock["moyenne_jour"]).round(1)
display(stock.round(1))

,moyenne_jour,p95_jour,max_jour,p95_sur_moyenne
AINS acétiques (M01AB),5.1,10.0,17.3,2.0
AINS propioniques (M01AE),4.0,7.7,14.5,1.9
Aspirine et dérivés (N02BA),3.9,8.0,16.0,2.1
Paracétamol (N02BE),30.5,60.4,161.0,2.0
Anxiolytiques (N05B),8.8,19.0,54.8,2.2
Somnifères (N05C),0.6,3.0,9.0,5.1
Antiasthmatiques (R03),5.6,20.0,45.0,3.6
Antihistaminiques (R06),2.9,8.0,15.0,2.7


Pourquoi : un jour sur vingt, la demande dépasse le p95. Le rapport p95/moyenne dit de combien de fois il faut sur-stocker (environ 2 pour les anti-inflammatoires, par exemple).

In [8]:
col = [c for c in groupes if "N02BE" in c][0]
display(d.nlargest(10, col)[["date", col]])      # les 10 plus gros jours

,date,Paracétamol (N02BE)
1093,2016-12-30,161.00
1107,2017-01-13,161.00
1094,2016-12-31,108.70
1831,2019-01-07,100.10
1851,2019-01-27,100.10
1850,2019-01-26,97.80
1102,2017-01-08,93.05
1844,2019-01-20,93.05
1018,2016-10-16,89.30
1011,2016-10-09,88.40
